In [1]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path("/Users/florence/team-data-visualization")
CLEAN = ROOT / "data/processed/us_accidents_clean"
OUT = ROOT / "data/processed"

want = ["State", "Weather_Group", "Start_Hour", "Severity",
        "Crossing", "Junction", "Railway", "Stop", "Traffic_Signal"]
avail = set(pq.ParquetFile(next(CLEAN.glob("*.parquet"))).schema.names)
cols = [c for c in want if c in avail]
print("missing columns:", [c for c in want if c not in avail])

df = pd.concat([pd.read_parquet(p, columns=cols)
                for p in sorted(CLEAN.glob("*.parquet"))], ignore_index=True)
print("rows:", len(df))                        # expect 7,610,699
print(df["Severity"].value_counts().sort_index())
# expect S1=65,570 S2=6,048,391 S3=1,295,235 S4=201,503

df["is_severe"] = (df["Severity"] >= 3).astype("int8")

grain = df.groupby(["State", "Weather_Group", "Start_Hour", "Severity"],
                   observed=True).size().reset_index(name="n")
grain.to_parquet(OUT / "dashboard_grain.parquet", index=False)
grain.to_csv(OUT / "dashboard_grain.csv", index=False)
print("grain rows:", len(grain))

road_cols = [c for c in want[4:] if c in df.columns]
road = (df.melt(id_vars=["Severity"], value_vars=road_cols,
                var_name="feature", value_name="present")
          .groupby(["feature", "present", "Severity"], observed=True)
          .size().reset_index(name="n"))
road.to_parquet(OUT / "dashboard_road.parquet", index=False)
road.to_csv(OUT / "dashboard_road.csv", index=False)
print("road rows:", len(road))

for name, keys in [("state_severe_share", ["State"]),
                   ("hour_pattern", ["Start_Hour"]),
                   ("weather_severe", ["Weather_Group"])]:
    t = df.groupby(keys)["is_severe"].agg(["mean", "size"]).reset_index()
    t.columns = keys + ["severe_share", "n"]
    t.sort_values("severe_share", ascending=False).to_csv(OUT / f"{name}.csv", index=False)

s = pd.read_csv(OUT / "state_severe_share.csv")
print("top state by severe share:")
print(s.head(3).to_string(index=False))
print("DONE — CSVs written")


missing columns: []
rows: 7610699
Severity
1      65570
2    6048391
3    1295235
4     201503
Name: count, dtype: int64
grain rows: 20268
road rows: 40
top state by severe share:
State  severe_share      n
   RI      0.463830  16851
   GA      0.443702 166159
   KY      0.428178  31968
DONE — CSVs written
